In [3]:
import json

FILE = "data/output_deepseek/03_deepseek_v4_flash_0731.json"

with open(FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

has_hallu_response = []
missing_hallu_response = []
result_none = []

for x in data:
    generation = x.get("deepseek_generation") or {}
    result = generation.get("result")

    # result = null
    if not isinstance(result, dict):
        result_none.append(x)
        missing_hallu_response.append(x)
        continue

    hallu = result.get("hallucinated_response")

    if isinstance(hallu, str) and hallu.strip():
        has_hallu_response.append(x)
    else:
        missing_hallu_response.append(x)

print("=" * 60)
print(f"Total samples             : {len(data):,}")
print(f"Has hallucinated_response : {len(has_hallu_response):,}")
print(f"Missing/empty response    : {len(missing_hallu_response):,}")
print(f"result = None/non-dict    : {len(result_none):,}")
print("=" * 60)

Total samples             : 13,970
Has hallucinated_response : 13,109
Missing/empty response    : 861
result = None/non-dict    : 861


# clean file output

In [2]:
import json

INPUT_FILE = "data/output_deepseek/03_deepseek_v4_flash_0731.json"
OUTPUT_FILE = "data/output_deepseek/deepseek_clean.json"


def get_hallu_span(result):
    """
    - intrinsic/extrinsic -> 1 hallucinated_span
    - mixed              -> list 2 hallucinated_span
    """

    target_label = result.get("target_label")

    # ==========================
    # MIXED
    # ==========================
    if target_label == "mixed_hallucination":
        components = result.get("mixed_components") or []

        spans = []

        if isinstance(components, list):
            for comp in components:
                if not isinstance(comp, dict):
                    continue

                span = (
                    comp.get("hallucinated_span")
                    or comp.get("hallu_span")
                )

                if span:
                    spans.append(span)

        elif isinstance(components, dict):
            # Phòng trường hợp mixed_components là dict
            for comp in components.values():
                if isinstance(comp, dict):
                    span = (
                        comp.get("hallucinated_span")
                        or comp.get("hallu_span")
                    )

                    if span:
                        spans.append(span)

        return spans

    # ==========================
    # INTRINSIC / EXTRINSIC
    # ==========================
    return result.get("hallucinated_span")


# ============================================================
# LOAD
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)


clean_data = []

for x in data:

    result = (x.get("deepseek_generation") or {}).get("result")

    # Bỏ generation fail
    if not isinstance(result, dict):
        continue

    hallu_answer = result.get("hallucinated_response")

    # Bỏ sample không có hallucinated response
    if not isinstance(hallu_answer, str) or not hallu_answer.strip():
        continue

    clean_data.append({
        "chunk_id": x.get("chunk_id"),
        "doc_id": x.get("doc_id"),
        "domain": x.get("domain"),
        "content": x.get("content"),
        "question": x.get("question"),
        "answer": x.get("answer"),

        "hallu_answer": hallu_answer.strip(),

        "target_label": result.get("target_label"),

        "hallu_span": get_hallu_span(result)
    })


# ============================================================
# SAVE
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        clean_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ============================================================
# REPORT
# ============================================================

print("=" * 60)
print(f"Original : {len(data):,}")
print(f"Clean    : {len(clean_data):,}")
print(f"Removed  : {len(data) - len(clean_data):,}")
print(f"Saved -> {OUTPUT_FILE}")

Original : 13,970
Clean    : 13,109
Removed  : 861
Saved -> data/output_deepseek/deepseek_clean.json


In [1]:
import json
from pathlib import Path

INPUT_FILE = Path("data/output_qwen/qwen3_8_output.json")
OUTPUT_FILE = Path("data/output_qwen/qwen_clean.json")


def get_generation_result(x):
    """
    Tìm result của Qwen.
    Hỗ trợ các tên phổ biến và fallback tự tìm *_generation.
    """
    possible_keys = [
        "qwen_generation",
        "qwen3_generation",
        "generation",
    ]

    for key in possible_keys:
        generation = x.get(key)
        if isinstance(generation, dict):
            result = generation.get("result")
            if isinstance(result, dict):
                return result

    # Fallback: tìm field dạng xxx_generation
    for key, value in x.items():
        if key.endswith("_generation") and isinstance(value, dict):
            result = value.get("result")
            if isinstance(result, dict):
                return result

    return None


def get_hallu_span(result):
    """
    intrinsic/extrinsic -> 1 span
    mixed               -> list các hallucinated span
    """
    target_label = result.get("target_label")

    if target_label in {
        "mixed_hallucination",
        "mixed_partial_hallucination",
    }:
        components = result.get("mixed_components") or []
        spans = []

        if isinstance(components, list):
            for comp in components:
                if not isinstance(comp, dict):
                    continue

                span = (
                    comp.get("hallucinated_span")
                    or comp.get("hallu_span")
                )

                if span:
                    spans.append(span)

        elif isinstance(components, dict):
            for comp in components.values():
                if not isinstance(comp, dict):
                    continue

                span = (
                    comp.get("hallucinated_span")
                    or comp.get("hallu_span")
                )

                if span:
                    spans.append(span)

        return spans

    return result.get("hallucinated_span")


# ============================================================
# LOAD
# ============================================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)


clean_data = []
failed_data = []

for x in data:

    result = get_generation_result(x)

    # Không có result hợp lệ
    if not isinstance(result, dict):
        failed_data.append(x)
        continue

    hallu_answer = result.get("hallucinated_response")

    # Không có hallucinated response
    if not isinstance(hallu_answer, str) or not hallu_answer.strip():
        failed_data.append(x)
        continue

    clean_data.append({
        "chunk_id": x.get("chunk_id"),
        "doc_id": x.get("doc_id"),
        "domain": x.get("domain"),
        "content": x.get("content"),
        "question": x.get("question"),
        "answer": x.get("answer"),
        "hallu_answer": hallu_answer.strip(),
        "target_label": result.get("target_label"),
        "hallu_span": get_hallu_span(result),
    })


# ============================================================
# SAVE CLEAN
# ============================================================

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(
        clean_data,
        f,
        ensure_ascii=False,
        indent=2,
    )


# ============================================================
# REPORT
# ============================================================

print("=" * 60)
print("QWEN CLEANING")
print("=" * 60)
print(f"Original : {len(data):,}")
print(f"Clean    : {len(clean_data):,}")
print(f"Failed   : {len(failed_data):,}")
print(f"Saved    : {OUTPUT_FILE}")
print("=" * 60)

QWEN CLEANING
Original : 14,883
Clean    : 595
Failed   : 14,288
Saved    : data/output_qwen/qwen_clean.json
